# Build an Agent from Scratch

Agentic Workflow -

- Agentic or agent workflow is when work is done iteratively in shorter phases instead of asking an LLM to accomplish a huge task in one shot.
- This type of iterative workflow gives a much better work product.
- Design pattern of Agentic workflows: 
    1. Planning - thinking through the steps needed for the task
    2. Tool use - knowing the type of tools needed and how to use them
    3. Reflection - iteratively improving result
    4. Multi-agent communication - agents in different steps working together
    5. Memory - Tracking progress and results

LangGraph -
- LangChain can also help to improve agentic workflows
- LangChain supports these cyclic graphs:-
    1. ReAct - models behave based on reason and act cycle 
    2. Self refine - it include iterative refinement with feedback
    3. Flow engineering - coding agents use flow engineering with preprocessing and code iterations
- LangChain can support LangGraphs which can help directly build agents
- Can build agents using just an LLM or LangGraph components

Useful for Building Agents -

1. Human input - agent should be able to receive human input to guide it at critical points.
2. Persistence - ability to store current state of information for returning to it later. This is useful for debugging agents and production analysing them.

### _Build Agent from Scratch with just an LLM:_

- We can build an agent with an LLM API and python code
- While building an agent, there are some jobs for the LLM and others managed by code around the LLM.
- A basic agent to build from scratch is based on the ReAct pattern
- ReAct patterns consists of reasoning and action. 
- An LLM thinks of what to do and then decides an action to execute. The action is executed in an environment and an observation is returned. Based on the observation, an LLM repeats the reason and action process.

#### _EXAMPLE Agent - ReAct Pattern:_

Import the things needed:

    import re
    from openai import OpenAI

Intialise the Language model:

    client = OpenAI()

Create class for Agent:

- Takes in system message/prompt as parameter
- Save the system msg as an attribute - self.system
- Track list of messages by appending all the messages in the ReAct loop. Append system message if it is there.
- Implement call method for the agent to take a msg that comes and append to the messages array. It also take the result from LLM and append into messages array with role of assistant.
- Implement execute method to call LLM model with messages

    class Agent:
        def __init__(self, system=""):
            self.system = system
            self.messages = []
            if self.system:
                self.messages.append({"role": "system", "content": system})

        def __call__(self, message):
            self.messages.append({"role": "user", "content": message})
            result = self.execute()
            self.messages.append({"role": "assistant", "content": result})
            return result

        def execute(self):
            completion = client.chat.completions.create(
                            model="gpt-4o", 
                            temperature=0,
                            messages=self.messages)
            return completion.choices[0].message.content
        

Create ReAct agent with a specific system message prompt:

- Ask LLM to loop through thought, action, pause, observation
- Output LLM result at end of the loop
- Specify the functions under though, action, pause and observation
- After observation result is received, output the answer


    prompt = """
    You run in a loop of Thought, Action, PAUSE, Observation.
    At the end of the loop you output an Answer
    Use Thought to describe your thoughts about the question you have been asked.
    Use Action to run one of the actions available to you - then return PAUSE.
    Observation will be the result of running those actions.

    Your available actions are:

    calculate:
    e.g. calculate: 4 * 7 / 3
    Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

    average_dog_weight:
    e.g. average_dog_weight: Collie
    returns average weight of a dog when given the breed

    Example session:

    Question: How much does a Bulldog weigh?
    Thought: I should look the dogs weight using average_dog_weight
    Action: average_dog_weight: Bulldog
    PAUSE

    You will be called again with this:

    Observation: A Bulldog weights 51 lbs

    You then output:

    Answer: A bulldog weights 51 lbs
    """.strip()

Define action functions:

    def calculate(what):
        return eval(what)

    def average_dog_weight(name):
        if name in "Scottish Terrier": 
            return("Scottish Terriers average 20 lbs")
        elif name in "Border Collie":
            return("a Border Collies average weight is 37 lbs")
        elif name in "Toy Poodle":
            return("a toy poodles average weight is 7 lbs")
        else:
            return("An average dog weights 50 lbs")

    known_actions = {
        "calculate": calculate,
        "average_dog_weight": average_dog_weight
    }

#### Manual Process of Running Agent -

Initialise Agent with system prompt:

    abot = Agent(prompt)

Call agent with query message:

    result = abot("How much does a toy poodle weigh?")
    print(result)

<b>
Result for this Query -

Thought: I should look up the average weight of a Toy Poodle using the average_dog_weight action.

Action: average_dog_weight: Toy Poodle

PAUSE
</b>

Follow the action and call the function:

    result = average_dog_weight("Toy Poodle")

Return result back to the agent under Observation:

    next_prompt = "Observation: {}".format(result)

    abot(next_prompt)

<b>
This will return Observation as final Answer -

'Answer: A Toy Poodle weighs an average of 7 lbs.'
</b>

#### Agent Execution Loop:

Automate agent loop using regex to parse LLM's response to determine if agent wants to take an action or output final answer:

    action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

Define Query function to run same ReAct process automatically:

- Add max turns to control how long it runs
- Use counter to count the iterations
- Intialise an agent to take system prompt and take the query as nect prompt
- In the loop, call agent and get result
- Then use regex created to parse the result and see if it is an action or final answer
- If the agent response contains action, make steps to complete actions. Use action and action input. If actions are not in known actions, raise exception. Get observation by doing the action from known actions dictionary and then call the function with the action input. Take next prompt as observation to the agent.
- Else, the agent response is the final answer and return it

    def query(question, max_turns=5):
        i = 0
        bot = Agent(prompt)
        next_prompt = question
        while i < max_turns:
            i += 1
            result = bot(next_prompt)
            print(result)
            actions = [
                action_re.match(a) 
                for a in result.split('\n') 
                if action_re.match(a)
            ]
            if actions:
                # There is an action to run
                action, action_input = actions[0].groups()
                if action not in known_actions:
                    raise Exception("Unknown action: {}: {}".format(action, action_input))
                print(" -- running {} {}".format(action, action_input))
                observation = known_actions[action](action_input)
                print("Observation:", observation)
                next_prompt = "Observation: {}".format(observation)
            else:
                return

##### _EXAMPLE - Call Agent with a Query:_

    question = """I have 2 dogs, a border collie and a scottish terrier. \
    What is their combined weight"""
    query(question)

<b>
The Process within the Agent Loop is shown below -

\
Thought: I need to find the average weight of both a Border Collie and a Scottish Terrier, then add them together to get the combined weight.

Action: average_dog_weight: Border Collie

PAUSE

 -- running average_dog_weight Border Collie

\
\
Observation: a Border Collies average weight is 37 lbs

Action: average_dog_weight: Scottish Terrier

PAUSE

 -- running average_dog_weight Scottish Terrier

\
\
Observation: Scottish Terriers average 20 lbs

Thought: Now that I have the average weights of both dogs, I can calculate their combined weight by adding the two values together.

Action: calculate: 37 + 20

PAUSE

 -- running calculate 37 + 20

 \
 \
Observation: 57

Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.

</b>